# As-of Joins on GPU

As-of joins are a foundational building block in time-series analytics, powering many of the most data-intensive workflows in finance and beyond.
Whether that be for trade-quote alignment, order book reconstruction, signal and feature alignment, protfolio valuation/risk snapshots, or IoT and sensor data synchronization, the KDB-X GPU Acceleration utilizes `.gpu.aj` to parallelize binary searches across every symbol simultaneously, using thousands of GPU cores.

In this tutorial, we demonstrate performance differences of joining trade and quote data on GPU vs CPU.

The core function we expose is `.gpu.aj` which is called as follows:
```q
.gpu.aj[c;t1;t2]
```
where
- `t1` is a table
- `t2` is a table
- `c` is a symbol list of n column names, common to `t1` and `t2`, and of matching type
- column `c[n]` is of a sortable type (typically time)

It's important to note that:
- columns in the list `c` may be mapped to the gpu for improved performance
- the list of values `c` can be of length 2 at most
- if `c` is of length 2, the only attribute supported on c[0] is the grouped attribute ``g#`

Data can be transferred between CPU and GPU as follows:
```q
.gpu.to t
```

To map only specific columns to the gpu (`time` and `sym` here), use `.gpu.xto`:
```q
.gpu.xto[`time`sym] t
```

For asof joins (`aj`):
- API automatically transfers to and from for CPU resident tables
- Better performance will be achieved by leaving target columns resident on GPU and using append for updates
- Limited to x2 columns

## Setup

This notebook is compatible with Linux, Mac, and Windows via WSL.

This notebook is designed to be run as a Python notebook, but the same functionality can be applied directly in a CLI with KDB-X (given it's running on a KDB-X CUDA application environment).
If running `q` from the terminal, just copy the `q` code and paste into your terminal.

We first want to download the required datasets and generate a database of trade and quote data.
We can configure the amount of data downloaded as a single day of NYSE TAQ files contain a huge amount of data - more on that [here](??).

Run `src/genHDB.sh` with the intended data configuration in order to download this data and generate an HDB using modified KX TAQ scripts (`src/tq.q`).
Build the HDB on an intended storage path, then [mount that directory to the Docker container at runtime](??)

In [1]:
import pykx as kx
import pandas as pd
import matplotlib.pyplot as plt
kx.q("\\s 48")

pykx.Identity(pykx.q('::'))

In [2]:
kx.Identity(kx.q('::'))

pykx.Identity(pykx.q('::'))

Now that we have loaded the necessary Python libraries and set PyKX values, we can enable 'q first' mode - this allows for running q code directly in Jupyter Notebook cells, similar to a q terminal

In [3]:
kx.util.jupyter_qfirst_enable()

KDB-X Python now running in 'jupyter_qfirst' mode. All cells by default will be run as q code.
Include '%%py' at the beginning of each cell to run as python code. 


Next we want to load in the GPU module as well as our HDB data (generated from genHDB.sh) that we've mounted to the Docker container at runtime:

In [4]:
.gpu:use`kx.gpu

\l /app/example/HDB/data

t:select from trade
q:select from quote

Let's take a look at our data - we can check table counts and take the first 3 records of each to get a look into what these tables look like

In [5]:
\c 2000 2000
show 3#t
count t
show 3#q
count q

569917
5007138
date       Time                 Exchange Symbol SaleCondition TradeVolume TradePrice TradeStopStockIndicator TradeCorrectionIndicator SequenceNumber TradeId SourceofTrade TradeReportingFacility ParticipantTimestamp TradeReportingFacilityTRFTimestamp TradeThroughExemptIndicator
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2025.10.02 0D04:04:26.715284597 Q        Z      @ TI          20          73.6       0                       0                        19438          ,"1"    N             0                      0D04:04:26.715270467                                    0                          
2025.10.02 0D04:04:26.715402540 K        Z      @ T           180         73.59      0                       0                        19439          ,"

Now we can look at the performance benefits of sending specific columns to GPU using `.gpu.xto` before performing as-of joins between our tables:

In [6]:
Tt:.gpu.xto[`Time`Symbol] t
Qq:.gpu.xto[`Time`Symbol] q

In [7]:
\t:5 aj[`Time;t;q]
\t:5 .gpu.aj[`Time;t;q]
\t:5 .gpu.aj[`Time;t;Qq]
\t:5 .gpu.aj[`Time;Tt;Qq]

126
54
74
60


In [8]:
\t:5 aj[`Symbol`Time;t;q]
\t:5 .gpu.aj[`Symbol`Time;t;q]
\t:5 .gpu.aj[`Symbol`Time;t;Qq]
\t:5 .gpu.aj[`Symbol`Time;Tt;Qq]

294
269
221
221


We can further dig into this by selecting specific columns and symbols from our tables and iterating many times over an as-of join:

In [9]:
// examples w/ selects
twentySyms:20#(select distinct Symbol from trade)`Symbol
freqSym: first twentySyms

a:select Time, TradePrice, TradeVolume, TradeStopStockIndicator, SaleCondition, Exchange from trade where Symbol=freqSym
b:select Time, Bid_Price, Offer_Price, Bid_Size, Offer_Size, Quote_Condition, Quote_Exchange: Exchange from quote where Symbol=freqSym

In [10]:
\t:50 aj[`Time;a;b]
\t:50 .gpu.aj[`Time;a;b]

274
31


In [11]:
a2:select Symbol, Time, TradePrice, TradeVolume, TradeStopStockIndicator, SaleCondition, Exchange from trade where Symbol in twentySyms
b2:select Symbol, Time, Bid_Price, Offer_Price, Bid_Size, Offer_Size, Quote_Condition, Quote_Exchange: Exchange from quote where Symbol in twentySyms

update `p#Symbol from `b2 // Needed to keep q aj fast

b2


In [12]:
\t:10 aj[`Symbol`Time;a2;b2]
\t:10 .gpu.aj[`Symbol`Time;a2;b2]

B:.gpu.to b2
\t:10 .gpu.aj[`Symbol`Time;a2;B]

Aa:.gpu.xto[`Time`Symbol] a2
\t:10 .gpu.aj[`Symbol`Time;Aa;B]

197
23
35
34
